# Stage 10.5 (Skid Grouping) — synthetic ground truth, Qwen zero-shot, GPU-only

No dataset has real skid labels (confirmed by direct inspection). Ground truth here is
CONSTRUCTED, not guessed: real P&ID equipment symbols get grouped by a deterministic script,
and a dashed boundary box is drawn directly on the image around each group - the actual
convention real P&ID drafters use to mark a vendor package. The dashed box IS the ground
truth, known by construction, no domain expertise required to grade it.

Also includes a renumbering metamorphic self-consistency check (no ground truth needed at
all): the same real boundaries, symbol index labels shuffled - a model reading the actual
drawn boundary (not pattern-matching on index values) should give the same partition either
way.

All CPU-side work (crop construction, boundary drawing, GPT-5.5-low scoring) already done
locally - GPT-5.5-low scored **91.9% pairwise accuracy** against real constructed ground
truth (n=12 crops), and was self-consistent under renumbering on 4/12 crops (most
disagreements were 1-3 symbols, not wholesale different answers). This notebook exists ONLY
to run the one GPU-dependent step: Qwen inference.

## 1. Config

In [1]:
HF_TOKEN = "PASTE_HF_TOKEN_HERE"
DATA_REPO = "timthy45/pnid-extraction-datasets"
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "paste-your-hf-token-here"


## 2. Install

In [2]:
!pip install -q -U transformers accelerate huggingface_hub

import torch
print("CUDA available:", torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 127.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 53.9 MB/s eta 0:00:00
CUDA available: True


## 3. Download the prepped package (crops + constructed ground truth + renumbered variants)

In [3]:
import zipfile, time, json, random, re
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA = Path("/content/data")
DATA.mkdir(exist_ok=True)

def fetch_with_retry(filename, max_attempts=20, base_backoff_s=10, max_backoff_s=90):
    last_err = None
    for attempt in range(max_attempts):
        try:
            return hf_hub_download(repo_id=DATA_REPO, filename=filename,
                                   repo_type="dataset", token=HF_TOKEN)
        except Exception as e:
            last_err = e
            wait = min(max_backoff_s, base_backoff_s * (1.5 ** attempt)) + random.uniform(0, 3)
            print(f"  [retry {attempt+1}/{max_attempts}] {type(e).__name__}: "
                 f"{str(e)[:120]} - waiting {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError(f"download of {filename} failed after {max_attempts} retries") from last_err

zip_path = fetch_with_retry("benchmarks/synth_skid_v1.zip")
PREP_DIR = DATA / "synth_skid"
PREP_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(PREP_DIR)

IMG_DIR = PREP_DIR / "images"
with open(PREP_DIR / "synth_meta.json") as f:
    meta = json.load(f)

# Qwen-specific override: the shared prompt (same one GPT-5.5-low was scored on, 91.9%
# accuracy) let Qwen collapse everything into one group on 5/12 crops - it wasn't using the
# dashed boundary signal at all, defaulting to "same drawing = same group." This version
# explicitly forbids that failure mode and states the default is separate, not merged.
PROMPT_TMPL = (
    "This P&ID crop has {n} symbols marked with numbered black boxes (0 to {nm1}).\n"
    "\n"
    "Some symbols - and ONLY some - are enclosed by a large DASHED rectangle labeled "
    "SKID-1 or SKID-2. Look carefully for a dashed (not solid) rectangle outline; it may "
    "be thin. ONLY symbols physically enclosed inside the same dashed rectangle belong to "
    "the same group.\n"
    "\n"
    "IMPORTANT - do not use any other reason to group symbols:\n"
    "- Being on the same pipe line, being connected by a line, or being near each other "
    "does NOT make two symbols the same group.\n"
    "- Being on the same drawing/sheet does NOT make symbols the same group.\n"
    "- The DEFAULT is that a symbol is its own group of one. Only put symbols together "
    "if they are BOTH inside the SAME dashed rectangle.\n"
    "- Do not group all symbols together. Most crops have 2 small dashed-boundary groups "
    "plus several standalone symbols outside any dashed boundary - not one big group.\n"
    "\n"
    "Respond with ONLY a JSON list of groups, each group a list of symbol numbers, e.g. "
    "[[0,1,2],[3,4],[5]]. Every symbol number 0 to {nm1} must appear exactly once."
)
records = meta["records"]
permutations = meta["permutations"]
print(f"loaded {len(records)} synthetic crops with constructed ground truth + renumbered variants")


benchmarks/synth_skid_v1.zip: reconstructing file:   0%|          |  0.00B / 4.56MB            

benchmarks/synth_skid_v1.zip: downloading bytes:           |  0.00B            

loaded 12 synthetic crops with constructed ground truth + renumbered variants


## 4. Load Qwen3-VL-8B base (zero-shot, no adapter)

In [4]:
from transformers import AutoModelForImageTextToText, AutoProcessor
import random as _random_retry

def load_with_retry(loader_fn, max_attempts=20, base_backoff_s=10, max_backoff_s=90):
    last_err = None
    for attempt in range(max_attempts):
        try:
            return loader_fn()
        except Exception as e:
            last_err = e
            backoff = min(max_backoff_s, base_backoff_s * (1.5 ** attempt))
            wait = backoff + _random_retry.uniform(0, backoff * 0.3)
            print(f"  [retry {attempt+1}/{max_attempts}] {type(e).__name__}: "
                 f"{str(e)[:160]} - waiting {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError(f"failed after {max_attempts} retries - HF Xet-bridge "
                       "signing issue persisted longer than usual; rerun this cell") from last_err

processor = load_with_retry(lambda: AutoProcessor.from_pretrained(QWEN_MODEL_ID))
model = load_with_retry(lambda: AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda")).eval()
print("Qwen base loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

def qwen_generate(image, prompt, max_tokens=500):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    t = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(t, skip_special_tokens=True).strip()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Qwen base loaded. VRAM: 17.5 GB


## 5. Score against constructed ground truth (the only GPU-dependent step)

In [5]:
from PIL import Image

def parse_groups(text, n):
    m = re.search(r"\[\s*\[.*\]\s*\]", text, re.S)
    if not m:
        return [[i] for i in range(n)]
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return [[i] for i in range(n)]

def groups_to_pair_labels(groups, n):
    group_of = {}
    for gi, g in enumerate(groups):
        for i in g:
            if 0 <= i < n:
                group_of[i] = gi
    labels = {}
    for i in range(n):
        for j in range(i + 1, n):
            labels[(i, j)] = (i in group_of and group_of.get(i) == group_of.get(j))
    return labels

total_pairs, correct_pairs = 0, 0
qwen_predictions = {}
for rec in records:
    sheet_id, n = rec["sheet_id"], rec["n"]
    img = Image.open(IMG_DIR / f"{sheet_id}.png")
    prompt = PROMPT_TMPL.format(n=n, nm1=n - 1)
    raw = qwen_generate(img, prompt)
    pred_groups = parse_groups(raw, n)
    qwen_predictions[sheet_id] = pred_groups
    gt_labels = groups_to_pair_labels(rec["gt_groups"], n)
    pred_labels = groups_to_pair_labels(pred_groups, n)
    for pair, gt_same in gt_labels.items():
        pred_same = pred_labels.get(pair, False)
        total_pairs += 1
        if pred_same == gt_same:
            correct_pairs += 1
    print(f"  {sheet_id} (n={n}): gt={rec['gt_groups']} pred={pred_groups}")

qwen_acc = correct_pairs / total_pairs if total_pairs else 0
print(f"\nQwen3-VL base vs CONSTRUCTED ground truth: {correct_pairs}/{total_pairs} pairs "
     f"= {qwen_acc:.1%} (n={len(records)} crops)")


  OPEN100_4_synth (n=9): gt=[[6, 0, 2, 3], [8, 5, 1], [7], [4]] pred=[[0, 1, 2, 3, 4, 5, 6, 7, 8]]
  OPEN100_3_synth (n=7): gt=[[5, 6], [4, 3, 2], [1], [0]] pred=[[0], [1, 2, 3, 4], [5, 6]]
  DatasetPID_79_synth (n=4): gt=[[1], [3], [2], [0]] pred=[[0], [1, 2], [3]]
  DatasetPID_379_synth (n=5): gt=[[2], [0, 3], [1], [4]] pred=[[0, 1, 2, 3, 4]]
  DatasetPID_176_synth (n=6): gt=[[4, 0], [3, 1], [5], [2]] pred=[[0, 1, 2, 3, 4], [5]]
  OPEN100_1_synth (n=7): gt=[[2, 3], [4, 0, 5], [1], [6]] pred=[[0, 1, 2, 3, 4], [5], [6]]
  OPEN100_0_synth (n=9): gt=[[0, 5, 3, 1], [2, 7, 8], [4], [6]] pred=[[0, 1, 3], [2, 7, 8], [4, 5, 6]]
  OPEN100_7_synth (n=8): gt=[[2, 4, 3], [0, 6, 5], [1], [7]] pred=[[0, 5], [1, 2, 3, 4, 6, 7]]
  OPEN100_11_synth (n=9): gt=[[8, 2, 5, 4], [3, 7, 0], [6], [1]] pred=[[0, 1, 2, 3, 4, 5, 6, 7, 8]]
  OPEN100_2_synth (n=8): gt=[[4, 1], [2, 3, 0, 7], [6], [5]] pred=[[0, 1, 2, 3, 4, 5, 6, 7]]
  OPEN100_5_synth (n=7): gt=[[6, 0], [3, 5, 4], [1], [2]] pred=[[0, 1, 2, 3, 4, 5, 

## 6. Renumbering self-consistency check (no ground truth needed)

In [6]:
consistent_count = 0
renumber_results = {}
for rec in records:
    sheet_id, n = rec["sheet_id"], rec["n"]
    orig_pred = qwen_predictions[sheet_id]
    orig_labels = groups_to_pair_labels(orig_pred, n)

    perm = permutations[sheet_id]["perm"]          # perm[old_index] = new_label
    inv_perm = {v: k for k, v in enumerate(perm)}  # new_label -> old_index

    img2 = Image.open(IMG_DIR / f"{sheet_id}_renumbered.png")
    prompt = PROMPT_TMPL.format(n=n, nm1=n - 1)
    raw2 = qwen_generate(img2, prompt)
    pred2_newlabels = parse_groups(raw2, n)
    pred2_canonical = [[inv_perm.get(x, x) for x in g] for g in pred2_newlabels]
    pred2_labels = groups_to_pair_labels(pred2_canonical, n)

    agree = sum(1 for pair in orig_labels if orig_labels[pair] == pred2_labels.get(pair, False))
    total = len(orig_labels)
    is_consistent = (agree == total)
    consistent_count += is_consistent
    renumber_results[sheet_id] = {
        "orig_pred": orig_pred, "renumbered_pred_mapped_back": pred2_canonical,
        "pair_agreement": f"{agree}/{total}", "fully_consistent": is_consistent,
    }
    print(f"  {sheet_id}: {'CONSISTENT' if is_consistent else 'inconsistent'} "
         f"({agree}/{total} pairs agree under renumbering)")

n_crops = len(records)
print(f"\nQwen3-VL base renumbering self-consistency: {consistent_count}/{n_crops} crops fully consistent")


  OPEN100_4_synth: inconsistent (22/36 pairs agree under renumbering)
  OPEN100_3_synth: inconsistent (16/21 pairs agree under renumbering)
  DatasetPID_79_synth: inconsistent (4/6 pairs agree under renumbering)
  DatasetPID_379_synth: CONSISTENT (10/10 pairs agree under renumbering)
  DatasetPID_176_synth: inconsistent (8/15 pairs agree under renumbering)
  OPEN100_1_synth: inconsistent (10/21 pairs agree under renumbering)
  OPEN100_0_synth: inconsistent (9/36 pairs agree under renumbering)
  OPEN100_7_synth: inconsistent (16/28 pairs agree under renumbering)
  OPEN100_11_synth: CONSISTENT (36/36 pairs agree under renumbering)
  OPEN100_2_synth: CONSISTENT (28/28 pairs agree under renumbering)
  OPEN100_5_synth: CONSISTENT (21/21 pairs agree under renumbering)
  DatasetPID_368_synth: inconsistent (10/15 pairs agree under renumbering)

Qwen3-VL base renumbering self-consistency: 4/12 crops fully consistent


## 7. Push results to HF, then disconnect

In [7]:
results = {
    "model": "qwen3vl-8b-base-zeroshot",
    "constructed_gt_accuracy": {"total_pairs": total_pairs, "correct_pairs": correct_pairs,
                               "pairwise_acc": qwen_acc, "n_crops": len(records)},
    "renumbering_consistency": {"n_crops": n_crops, "fully_consistent": consistent_count,
                                "per_crop": renumber_results},
    "predictions": qwen_predictions,
}
with open("/content/qwen_synth_skid_results.json", "w") as f:
    json.dump(results, f, indent=2)

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj="/content/qwen_synth_skid_results.json",
    path_in_repo="benchmarks/qwen_synth_skid_results_v2_qwen_prompt.json",
    repo_id=DATA_REPO, repo_type="dataset", token=HF_TOKEN)
print("results pushed to HF")

from google.colab import runtime
runtime.unassign()


results pushed to HF
